# 02 · Module 2: Credit risk

**Question.** Which loans to approve, review or reject, and what the approved book earns after expected credit loss.

The strongest module (real signal, 20% default rate): build it first and use it as the template.

**Data.** `nova_loans.csv`: 5,000 loans in rupees (book about ₹640 cr), Default_Flag 20%. Split: repeated stratified 5-fold CV, 20% hold-out.

**Professor's anchor.** S8 (PD, LGD, EAD, scoring, green/amber/red thresholds as a business choice) and TA4 (the MSME early-warning engagement).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from novakit import ASSUMPTIONS, CRORE, LAKH, SEED, load, use_module
from novakit import evaluate, explain, finance, handoff, splits

RES, FIG = use_module("02")

In [ ]:
loans = load("loans")
loans.head()

> Evaluation standard: a naive baseline and a transparent model before any tree; PR-AUC against the base rate, Brier score and a calibration check for classifiers; the threshold chosen on validation data and reported on test data; SHAP or coefficients for the model that drives the decision; a stability check by period, segment or stock.

## 1. Features

Loan-to-income, collateral cover (Collateral_Value ÷ Loan_Amount), total leverage ((Existing_Loan + Loan_Amount) ÷ Annual_Income), instalment-to-income (`finance.monthly_instalment`), Customer_Type as a flag.

## 2. Interest_Rate check

Train with and without Interest_Rate. If it carries most of the signal it is the lender's own risk pricing and the model mostly repeats it. Report both; say which you would deploy.

## 3. Models

Logistic regression (the scorecard baseline) against random forest and XGBoost, with class weights. `splits.repeated_cv()`, 20% hold-out.

## 4. Metrics and calibration

`evaluate.classification_metrics` (ROC-AUC, PR-AUC against 20%, KS, Brier) and `evaluate.calibration_table`. Calibrate with isotonic or Platt: ECL needs true probabilities.

## 5. PD output (Task 2)

Customer_ID, Predicted_PD, Risk_Category. Check the brief's bands (<5%, 5–15%, 15–30%, >30%) against the PD distribution and the default rate in each; adjust if a band is nearly empty.

## 6. ECL (Task 3)

ECL = PD × 40% × Loan_Amount (`finance.ecl`). Sensitivity: `finance.collateral_lgd` (20% when cover ≥ 1.5, 60% when cover < 0.5).

## 7. Profit per loan

Expected annual margin = (Interest_Rate − 6.5% funding cost) × Loan_Amount − annual ECL. Show the approve threshold that maximises portfolio profit, then the more conservative one you recommend and why (risk appetite and capital, S4).

## 8. Decision

**Approve:** Low or Medium band and margin positive after ECL. **Review:** High band; or a top-decile loan at any PD; or collateral cover < 1 with PD above 10%. **Reject:** Very High band, or margin negative after ECL.

## Handoff to Module 9

Approved book size, yield, ECL rate and RAROC, separately for corporate and retail (capital charge 100% and 75%).

In [ ]:
# lines = [handoff.make_line(
#     business_line="...", capacity_cr=..., expected_return_pct=..., expected_loss_pct=...,
#     capital_charge_pct=..., stress_loss_pct=..., ml_edge="validated | weak | none", evidence="...",
#     key_risks=[("risk", "control"), ("risk", "control")],
# )]
# handoff.write_handoff("02", lines)

## Traps to check before handing over

An uncalibrated PD makes ECL wrong even when AUC is fine. The default label has no time stamp, so the model cannot be tested across credit cycles: say so.